# 03. Mercado de renda fixa (CDI)
Desenvolvimento da classe RendaFixa, que é quem fornece a taxa livre de risco. Requisito F3.

In [ ]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

import pandas as _pd
print(f'kernel: {sys.executable}  (pandas {_pd.__version__})')
if 'venv' not in sys.executable.replace(os.sep, '/').split('/'):
    print('  ATENCAO: este kernel NAO e o venv do projeto; os resultados podem diferir.')


kernel: C:\Users\DELL\Desktop\TCC\optimal-trading-strategies-stochastic-discrete-time-environmente-brazil\venv\Scripts\python.exe  (pandas 2.2.3)


## Desenvolvimento

A classe abaixo nasceu neste notebook. Depois que o teste passou, ela foi para app/mercado.py.

In [2]:
class RendaFixa:
    """Mercado de renda fixa (CDI) — fornece a taxa livre de risco. (F3)"""

    def __init__(self, cdi_anual: float, periodos_por_ano: int = 12) -> None:
        """
        Recebe o CDI anual em decimal (0.10 para 10% ao ano) e em quantos
        periodos o ano e dividido: 12 se a base for mensal, 252 se for diaria.

        O cdi_anual tem que ser maior que -1 e o periodos_por_ano, pelo menos
        1; fora disso a conversao do retorno_livre_risco nao esta definida.
        """
        cdi_anual = float(cdi_anual)
        periodos_por_ano = int(periodos_por_ano)
        if cdi_anual <= -1.0:
            raise ValueError(
                f"cdi_anual deve ser > -1; veio {cdi_anual:g}. A conversão "
                "eleva (1 + cdi_anual) a 1/periodos_por_ano, e com a base "
                "negativa o resultado sai complexo. "
            )
        if periodos_por_ano < 1:
            raise ValueError(
                f"periodos_por_ano deve ser >= 1; veio {periodos_por_ano}."
            )
        self.cdi_anual = cdi_anual
        self.periodos_por_ano = periodos_por_ano

    def retorno_livre_risco(self) -> float:
        """A taxa livre de risco de um periodo, liquida. (F3)

        A conversao de ano pra periodo e composta, e nao dividindo por 12:

            R_f = (1 + cdi_anual) ** (1 / periodos_por_ano) - 1

        Por exemplo, 10% ao ano no mensal da (1.10) ** (1/12) - 1, que e mais
        ou menos 0.007974. Na hora de propagar a riqueza usa-se 1 + R_f.
        """
        return (1.0 + self.cdi_anual) ** (1.0 / self.periodos_por_ano) - 1.0

**Teste**: converter o CDI anual para o R_f mensal.

In [3]:
rf = RendaFixa(0.10).retorno_livre_risco()

print('CDI 10% a.a. -> R_f mensal =', rf)

CDI 10% a.a. -> R_f mensal = 0.007974140428903764


In [4]:
assert abs(rf - ((1.10)**(1/12) - 1)) < 1e-15

In [5]:
# diario: 252 pregoes/ano (ida-e-volta exata)
rf_d = RendaFixa(0.1312, 252).retorno_livre_risco()

print('CDI 13.12% a.a. -> R_f diario =', rf_d)

CDI 13.12% a.a. -> R_f diario = 0.0004893221241111245


In [ ]:
assert abs((1 + rf_d) ** 252 - 1.1312) < 1e-12
assert rf_d < rf

**Teste**: o domínio da conversão — `cdi_anual > -1` e `periodos_por_ano >= 1`.

In [7]:
print('sem a guarda, (1 + (-1.5)) ** (1/252) =', (1.0 + (-1.5)) ** (1.0 / 252))
print()

for ruim in (-1.0, -1.5):
    try:
        RendaFixa(ruim, 252)
        print(f'cdi_anual={ruim}: PASSOU (nao devia)')
    except ValueError as e:
        print(f'cdi_anual={ruim}: recusado -> {str(e).split(". ")[0]}')

sem a guarda, (1 + (-1.5)) ** (1/252) = (0.9971757012688457+0.012432072064397498j)

cdi_anual=-1.0: recusado -> cdi_anual deve ser > -1; veio -1
cdi_anual=-1.5: recusado -> cdi_anual deve ser > -1; veio -1.5


In [8]:
try:
    RendaFixa(0.10, 0)
except ValueError as e:
    print('periodos_por_ano=0: recusado ->', e)

periodos_por_ano=0: recusado -> periodos_por_ano deve ser >= 1; veio 0.


In [9]:
for ruim in (-1.0, -1.5, -10.0):
    try:
        RendaFixa(ruim, 252)
        raise AssertionError(f'cdi_anual={ruim} passou, e nao devia')
    except ValueError:
        pass

In [10]:
for ruim in (0, -1, -12):
    try:
        RendaFixa(0.10, ruim)
        raise AssertionError(f'periodos_por_ano={ruim} passou, e nao devia')
    except ValueError:
        pass

In [11]:
assert RendaFixa(-0.99, 252).retorno_livre_risco() < 0